# 01 Template — 10x Genomics 单细胞数据读入 + 质量控制

本 notebook 是接入 10x mtx 格式新数据集的通用模板，以 Kim（`01_kim.ipynb`，类器官）
和 Nancang（`01_nancang.ipynb`，活检组织）为参照实现，对齐两者的全部核心机制：
expression_contract、run_contract、preflight 校验、双细胞三态定级等。

## 使用方式

1. 修改 **PARAMS cell** 中的 `DATA_SOURCE_ID`、`RAW_MATRIX_DIR` 及 `OUTPUT_VERSION`
2. 按需调整 QC 阈值（`N_MAD`、`MIN_GENES` 等）和方法开关
3. 从头到尾运行全部 cell
4. 检查 QC 图、MAD 敏感度曲线、双细胞诊断表
5. checkpoint cell 自动将结果 promote 为正式产物（或 NEEDS_REVIEW 时保留 draft 待 PI 审阅）

## 与 Kim/Nancang 的关系

- Kim（类器官）：N_MAD=3，SOUPX_ENABLED=False，FLAG_HEMOGLOBIN=False
- Nancang（活检组织）：N_MAD=5，SOUPX_ENABLED=True，FLAG_HEMOGLOBIN=True
- **本模板**：提供完整骨架与全部方法实现；具体参数值需按数据集特点调整

## 适用范围

- 10x Genomics 单细胞转录组（mtx 格式，`filtered_feature_bc_matrix`）
- 单样本或多样本（自动发现子目录并拼接）
- 如需接入 h5ad 格式数据，参考 Kim notebook 中 `sc.read_h5ad()` 的用法


## 唯一参数入口（四组）

本 cell 是运行前唯一需要修改参数的地方，分四组：

- **组一 · 数据源**：指定哪批数据、临床元数据、manifest 路径
- **组二 · QC 阈值**：改变纳入/排除哪些细胞和基因的科研判断
- **组三 · 方法开关**：开/关 SoupX 环境 RNA 校正、双细胞检测、细胞周期评分
- **组四 · 输出标识**：RUN_ID 与版本号，用于复现追溯

每个参数下方注释给出四要素：**是什么 / 默认依据 / 调大调小影响 / 何时该改**。
改参数后请新开 RUN_ID（bump `OUTPUT_VERSION` 或修改 `RUN_ID` 后缀），不覆盖旧 run。


In [ ]:
# === PARAMS：本 cell 是唯一修改参数的地方 ===

# ══ 组一 · 数据源 ═══════════════════════════════════════════
# 是什么：数据集唯一标识，用于输出文件命名和 source_dataset 列
# 默认依据：无默认值，必须由用户填入（如 "kim"、"nancang"）
# 影响：贯穿全流程——输出文件名、RUN_ID、obs["source_dataset"] 均以此命名
# 何时该改：接入新数据集时设定，后续不轻易改动
DATA_SOURCE_ID   = "dataset_id"

# 是什么：10x cellranger 输出的 filtered_feature_bc_matrix 目录绝对/相对路径
# 默认依据：无默认值，必须由用户填入实际路径
# 影响：路径不存在时 preflight 会提前报错，不会跑到一半才崩
# 何时该改：每次接入新数据集时修改
RAW_MATRIX_DIR   = "data/raw/dataset_id/filtered_feature_bc_matrix"

# 是什么：临床元数据 CSV 文件路径（可选）
# 默认依据："" 表示无临床数据，跳过 obs 合并
# 影响：填入路径后会自动合并临床信息到 adata.obs
# 何时该改：有临床随访/表型数据时填入路径
CLINICAL_CSV     = ""

# 是什么：数据集的 manifest.yaml 路径（声明输入格式、样本信息、预处理历史）
# 默认依据：按 DATA_SOURCE_ID 命名，放在 manifests 目录下
# 影响：manifest 不存在时会跳过 manifest-driven 校验（preprocessing_done 等）
# 何时该改：manifest 不存在时先创建，或设为 "" 跳过
MANIFEST_YAML    = "notebooks/01_per_dataset/manifests/dataset_id.yaml"

# ══ 组二 · QC 阈值 ═════════════════════════════════════════
# 是什么：QC 阈值策略，adaptive（数据驱动跨样本自适应）或 fixed（固定阈值）
# 默认依据：adaptive，利用数据自身分布自动调整，无需预知数据基线
# 调大调小影响：fixed = 使用下方四个固定阈值，适合已知标准的数据
# 何时该改：有实验室既定阈值或类器官数据时切 fixed
QC_STRATEGY = "adaptive"

# 是什么：自适应阈值的中位绝对偏差（MAD）倍数
# 默认依据：5，scRNA-seq 常用宽松界，优先保留更多细胞
# 调大调小影响：调大=更宽松、留更多细胞（含低质量）；调小=更严、多删细胞（含真细胞）
# 何时该改：活检组织 3-4，类器官 5-7；污染重时调小，稀有细胞多时调大
N_MAD = 5

# 是什么：是否按 sample_id 分组独立计算 MAD（per-sample mode）
# 默认依据：True，各样本测序深度和质量不同，独立阈值更公允
# 影响：False 回退全局阈值（可能被高质量样本拉偏整体分布）
# 何时该改：样本极少（<3）或需与旧全局行为严格对齐时设 False
PER_SAMPLE_MAD = True

# 是什么：基因保留的最低检出细胞数阈值
# 默认依据：3，去除噪声基因的常用值，平衡真实信号保留与噪声过滤
# 调大调小影响：调大=更激进去基因，可能丢失稀有细胞类型 marker；调小=保留更多基因但噪声增多
# 何时该改：关注稀有细胞类型时下调至 1-2
MIN_CELLS_PER_GENE = 3

# --- 固定阈值（仅 QC_STRATEGY="fixed" 时生效）---
# 是什么：fixed 模式下 n_genes 下界，过滤基因数过低的空液滴/死细胞
# 默认依据：200，常规单细胞经验值
# 影响：仅 QC_STRATEGY="fixed" 生效，adaptive 下为占位不参与
# 何时该改：切到 fixed 模式且有明确标准时按需调整
MIN_GENES = 200

# 是什么：fixed 模式下 n_genes 上界，过滤可能的双细胞
# 默认依据：6000，常规单细胞经验值；None 表示不设上限
# 影响：同上，仅 fixed 生效
# 何时该改：高测序深度数据上调
MAX_GENES = None

# 是什么：fixed 模式下 total_counts 下界，过滤 UMI 数过低的细胞
# 默认依据：500，常规单细胞经验值
# 影响：同上，仅 fixed 生效
# 何时该改：低深度测序数据下调
MIN_COUNTS = 500

# 是什么：fixed 模式下线粒体百分比上界，过滤细胞膜破损/凋亡细胞
# 默认依据：20%，常规单细胞经验值
# 影响：同上，仅 fixed 生效
# 何时该改：高代谢活性组织（心肌等）上调
MAX_PCT_MT = 20

# ══ 组三 · 方法开关 ═══════════════════════════════════════
# 是什么：是否启用 SoupX 环境 RNA 校正
# 默认依据：False，多数新数据集不提供 raw droplets 矩阵，先关后按需开
# 影响（True）：要求 manifest 中提供 raw_matrix_path，生成 layers["counts_soupx"]
# 影响（False）：跳过 SoupX，以未校正 counts 继续
# 何时该改：具备完整 raw droplets + filtered 矩阵时设为 True
SOUPX_ENABLED = False

# 是什么：counts_soupx 层是否对校正值取整
# 默认依据：True，部分下游 DEG 方法要求整数输入
# 影响（False）：保留 SoupX 非整数校正值，适合连续值分析
# 何时该改：下游只用连续值方法时设 False
SOUPX_INTEGER_ROUND = True

# 是什么：是否运行双细胞检测（scrublet per-sample）
# 默认依据：True，双细胞干扰细胞类型注释，默认应检测
# 影响（False）：所有细胞标记 singlet，不运行 scrublet
# 何时该改：已知数据无双细胞问题（如已通过其他方法去除）或样本量极少时设 False
DOUBLET_DETECTION = True

# 是什么：scrublet 预期双细胞率
# 默认依据：0.05，10x 常规通量约 0.8%/千细胞（5000 细胞 ≈ 4-5%）
# 影响：偏高使更多细胞被评为 doublet
# 何时该改：按上机细胞投入量调整，高负载时上调
EXPECTED_DOUBLET_RATE = 0.05

# 是什么：旧手动双细胞阈值（向后兼容），None 时优先使用 scrublet 自动阈值
# 默认依据：None，优先用 scrublet 自动阈值
# 影响：float 值直接覆盖自动阈值
# 何时该改：自动阈值明显不合理时手动指定
DOUBLET_SCORE_THRESHOLD = None

# 是什么：三态定级的高置信 doublet 阈值上限
# 默认依据：None，优先用 scrublet 自动阈值 threshold_
# 影响：高于此值标记为 doublet（默认排除）；过高漏检 doublet
# 何时该改：双峰不明显或自动阈值不合理时手动指定
DOUBLET_SCORE_HIGH = None

# 是什么：三态定级的低置信 singlet 上界
# 默认依据：None，自动派生为 HIGH×0.5
# 影响：低于此值标记为 singlet；过低可能误删 singlet
# 何时该改：对纯度要求高时上调 LOW 以扩大 uncertain 区间
DOUBLET_SCORE_LOW = None

# 是什么：uncertain 细胞是否纳入下游分析
# 默认依据：True，只默认排除高置信 doublet，边界细胞标记但保留
# 影响（False）：连 uncertain 一起排除，更保守但损失数据量
# 何时该改：对纯度要求极高时设 False
DOUBLET_UNCERTAIN_INCLUDE = True

# 是什么：单样本 doublet 预测比例上限告警阈值
# 默认依据：0.40，超出此值触发 needs_review
# 影响：触发 needs_review 要求 PI 审阅 per-sample 诊断表
# 何时该改：不同组织类型预期双率不同时调整
DOUBLET_RATE_ALERT_HIGH = 0.40

# 是什么：单样本 doublet 预测比例下限告警阈值（可选）
# 默认依据：None，不设下限告警
# 影响：float 值低于此值触发 needs_review
# 何时该改：需要检测异常低 doublet 率时设置
DOUBLET_RATE_ALERT_LOW = None

# 是什么：可稳定计算 doublet 阈值的样本最低细胞数
# 默认依据：50，低于此数阈值估计不可靠，记 needs_review
# 影响：低于此数该样本全部标记 singlet（不强行算阈值）
# 何时该改：样本普遍偏小时下调（注意阈值可能不稳）
DOUBLET_MIN_CELLS = 50

# 是什么：是否标记血红蛋白基因（仅标记不删除）
# 默认依据：True，消化道活检中 HB 污染信息对下游有参考价值
# 影响（False）：关闭则不产出 flag_hb 列与 pct_counts_hb 指标
# 何时该改：确信无红细胞污染问题（如类器官数据）时关闭
FLAG_HEMOGLOBIN = True

# 是什么：是否标记应激基因（IEGs + HSPs，组织解离诱导）
# 默认依据：True，解离应激信息对下游质控判断有参考价值
# 影响（False）：关闭则不产出 pct_counts_stress 指标
# 何时该改：不关心解离应激信号时关闭
FLAG_STRESS_GENES = True

# 是什么：是否做细胞周期评分（Tirosh 2015 marker genes）
# 默认依据：True，细胞周期是重要的技术协变量，下游可据此回归
# 影响（False）：关闭则跳过评分，不做周期回归时节省计算
# 何时该改：不做周期回归或数据与周期无关时关闭
CELL_CYCLE_SCORING = True

# ══ 组四 · 输出标识 ═══════════════════════════════════════
# 是什么：输出版本号，每次调参建议递增
# 默认依据："v1"，首次运行从 v1 开始
# 影响：同时影响 RUN_ID 和输出文件名
# 何时该改：每次调参重新运行时 bump（v1→v2→v3）
OUTPUT_VERSION = "v1"

# 是什么：运行唯一标识，用于 run_contract 管理（manifest + checkpoint）
# 默认依据：{数据集}_{版本}，自动派生
# 影响：同一 RUN_ID 不可覆盖已有 run（prepare_run 拒绝重复）
# 何时该改：调参后 bump OUTPUT_VERSION 即可自动更新
RUN_ID = f"01-{DATA_SOURCE_ID}-{OUTPUT_VERSION}"

# 是什么：run 结果存放根目录
# 默认依据：results/runs，项目统一约定
# 影响：每个 RUN_ID 在下方创建独立子目录
# 何时该改：一般不修改
RUN_ROOT = "results/runs"

# 是什么：输出 h5ad 文件名
# 默认依据：{数据集}_{版本}.h5ad，便于追溯
# 影响：checkpoint cell 写入此文件名
# 何时该改：一般不单独修改，跟随 OUTPUT_VERSION
OUTPUT_FILENAME = f"01_{DATA_SOURCE_ID}_{OUTPUT_VERSION}.h5ad"

# 是什么：随机种子，影响 scrublet 与任何随机过程的可复现性
# 默认依据：42
# 影响：不同种子可能产生略有不同的 doublet 评分
# 何时该改：需要精确复现时保持固定
RANDOM_SEED = 42


## Preflight：运行前配置校验

在读入数据前集中校验路径、输入文件、参数取值与方法开关的合法性。
任一不合法立即 raise 明确错误，避免跑到中途才崩或静默产出错误结果。

校验内容包括：
- RAW_MATRIX_DIR 路径是否存在
- MANIFEST_YAML 文件是否存在（若提供）
- QC_STRATEGY 是否为合法值
- N_MAD 是否为正数
- SOUPX_ENABLED=True 时是否具备 R 环境
- DOUBLEt 参数逻辑合法性（LOW < HIGH 等）


In [ ]:
# === Preflight：运行前集中校验 ===
# 在读入大矩阵之前校验路径、输入与参数，任何非法值在此 raise
import os, sys, yaml
from pathlib import Path

_errors = []
_warnings = []

# ---- 1. 数据路径存在性 ----
_raw_dir = Path(RAW_MATRIX_DIR)
if not _raw_dir.exists():
    _errors.append(f"RAW_MATRIX_DIR 不存在: {RAW_MATRIX_DIR}")
else:
    # 检查是否含 10x mtx 文件
    _has_mtx = any(
        (_raw_dir / "matrix.mtx").exists(),
        (_raw_dir / "matrix.mtx.gz").exists(),
    ) or any(
        (sub / "matrix.mtx").exists() or (sub / "matrix.mtx.gz").exists()
        for sub in _raw_dir.iterdir() if sub.is_dir() and not sub.name.startswith(".")
    )
    if not _has_mtx:
        _errors.append(f"RAW_MATRIX_DIR 中未找到 matrix.mtx(.gz): {RAW_MATRIX_DIR}")

# ---- 2. manifest 文件存在性（若提供）----
_manifest_exists = MANIFEST_YAML and Path(MANIFEST_YAML).exists()
if MANIFEST_YAML and not _manifest_exists:
    _warnings.append(f"MANIFEST_YAML 不存在: {MANIFEST_YAML}，將跳过 manifest-driven 校验")

# ---- 3. DATA_SOURCE_ID 非占位符 ----
if DATA_SOURCE_ID == "dataset_id":
    _errors.append("DATA_SOURCE_ID 仍为占位值 'dataset_id'，请填入实际数据集标识")

# ---- 4. QC 策略合法性 ----
if QC_STRATEGY not in ("adaptive", "fixed"):
    _errors.append(f"QC_STRATEGY 必须为 'adaptive' 或 'fixed'，当前: {QC_STRATEGY}")

# ---- 5. N_MAD 合法 ----
if not isinstance(N_MAD, (int, float)) or N_MAD <= 0:
    _errors.append(f"N_MAD 必须为正数，当前: {N_MAD}")

# ---- 6. doublet 参数逻辑 ----
if DOUBLET_SCORE_HIGH is not None and DOUBLET_SCORE_LOW is not None:
    if DOUBLET_SCORE_LOW >= DOUBLET_SCORE_HIGH:
        _errors.append(
            f"DOUBLET_SCORE_LOW ({DOUBLET_SCORE_LOW}) 必须 < DOUBLET_SCORE_HIGH ({DOUBLET_SCORE_HIGH})"
        )
if DOUBLET_SCORE_THRESHOLD is not None and DOUBLET_SCORE_THRESHOLD <= 0:
    _errors.append(f"DOUBLET_SCORE_THRESHOLD 必须 > 0，当前: {DOUBLET_SCORE_THRESHOLD}")

# ---- 7. SoupX 前置条件（若启用）----
_r_available = False
if SOUPX_ENABLED:
    import subprocess
    try:
        _r_result = subprocess.run(
            ["Rscript", "--version"], capture_output=True, text=True, timeout=5
        )
        _r_available = _r_result.returncode == 0
    except (FileNotFoundError, subprocess.TimeoutExpired):
        _r_available = False
    if not _r_available:
        _warnings.append("SOUPX_ENABLED=True 但 Rscript 不可用，SoupX 将被跳过")

R_AVAILABLE = _r_available
RSCRIPT_BIN = "Rscript"

# ---- 汇总 ----
print("=" * 60)
print("Preflight 校验结果")
print("=" * 60)
print(f"  DATA_SOURCE_ID : {DATA_SOURCE_ID}")
print(f"  RAW_MATRIX_DIR  : {RAW_MATRIX_DIR} ({"存在" if _raw_dir.exists() else "不存在"})")
print(f"  MANIFEST_YAML   : {MANIFEST_YAML} ({"存在" if _manifest_exists else "不存在"})")
print(f"  QC_STRATEGY     : {QC_STRATEGY}")
print(f"  N_MAD           : {N_MAD}")
print(f"  SOUPX_ENABLED   : {SOUPX_ENABLED} (R 可用: {_r_available})")
print(f"  DOUBLET_DETECTION: {DOUBLET_DETECTION}")
print(f"  CELL_CYCLE_SCORING: {CELL_CYCLE_SCORING}")
print(f"  OUTPUT_VERSION  : {OUTPUT_VERSION}")
print(f"  RUN_ID          : {RUN_ID}")
if _errors:
    for e in _errors:
        print(f"  [ERROR] {e}")
if _warnings:
    for w in _warnings:
        print(f"  [WARN]  {w}")
print("=" * 60)
if _errors:
    raise RuntimeError(f"Preflight 校验失败: {len(_errors)} 个错误")
print("Preflight 校验通过，开始执行。")


In [ ]:
# === Setup：脚手架 + 全部依赖导入 ===
# 一行替代原来 20 行样板代码：找项目根 + BLAS 线程设置 + sys.path 注入
# 遵循 "platform 决策可见" 原则——打印所有决策结果
from scrna_integration.bootstrap import init
_root = init()

# ---- 第三方依赖 ----
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, gc
from scipy.stats import median_abs_deviation

# ---- 项目内部模块 ----
from scrna_integration.io import sync_gene_ids
from scrna_integration.run_contract import (
    prepare_run, promote_run, sha256_file, atomic_write_json,
    determine_stage_status, snapshot_effective_parameters,
    collect_runtime_provenance, validate_expression_contract,
    MethodStatus, StageStatus,
)
from scrna_integration.per_dataset_schema import validate_per_dataset_output

# ---- 全局设置 ----
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=True)
sns.set_context("notebook")
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(RANDOM_SEED)

print(f"\n项目根: {_root}")
print(f"RUN_ID : {RUN_ID}")
print(f"输出文件: {OUTPUT_FILENAME}")


In [ ]:
# === 数据读入：10x mtx 格式，多子目录自动发现 ===
# 替代原来的 read_with_manifest——按透明性铁律，数据读取逻辑拆回 cell
# 每个 notebook 只写自己数据集所需的那一种格式

import anndata

# ---- 读取 manifest（若存在）获取 source_dataset 等元信息 ----
_manifest_data = {}
if _manifest_exists:
    with open(MANIFEST_YAML) as f:
        _manifest_data = yaml.safe_load(f)
    source_dataset = str(_manifest_data.get("source_dataset", DATA_SOURCE_ID))
    print(f"从 manifest 读取: source_dataset={source_dataset}")
else:
    source_dataset = DATA_SOURCE_ID
    print(f"无 manifest，source_dataset={source_dataset}")

# ---- 发现所有 10x mtx 子目录 ----
data_dir = Path(RAW_MATRIX_DIR)
sub_dirs = []
for sub in sorted(data_dir.iterdir()):
    if sub.is_dir() and not sub.name.startswith("."):
        if (sub / "matrix.mtx").exists() or (sub / "matrix.mtx.gz").exists():
            sub_dirs.append(sub)

# 若没有子目录含 mtx，检查当前目录本身是否为 mtx 目录
if not sub_dirs:
    if (data_dir / "matrix.mtx").exists() or (data_dir / "matrix.mtx.gz").exists():
        sub_dirs.append(data_dir)

if not sub_dirs:
    raise FileNotFoundError(f"{RAW_MATRIX_DIR} 中未找到任何 matrix.mtx(.gz) 文件")
print(f"发现 {len(sub_dirs)} 个 10x 数据子目录: {[d.name for d in sub_dirs]}")

# ---- 逐个子目录读取 10x mtx ----
adatas = []
for sub_dir in sub_dirs:
    adata_sub = sc.read_10x_mtx(sub_dir, var_names="gene_symbols")
    # 为每个 barcode 加样本前缀，避免拼接时 barcode 冲突
    adata_sub.obs_names = [f"{sub_dir.name}_{bc}" for bc in adata_sub.obs_names]
    adata_sub.obs["source_dataset"] = source_dataset
    adata_sub.obs["sample_id"] = sub_dir.name
    adatas.append(adata_sub)
    print(f"  {sub_dir.name}: {adata_sub.n_obs:,} 细胞 x {adata_sub.n_vars:,} 基因")

# ---- 多子目录拼接（单目录直接使用）----
if len(adatas) > 1:
    adata = anndata.concat(adatas, join="outer", index_unique="_")
    print(f"\n拼接完成: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
else:
    adata = adatas[0]

# ---- 基因 ID 同步：symbol → ensembl_id ----
# var.index 为 gene symbol，补充 var["ensembl_id"] 列
# 这是 02_merged inner join 的前提——所有数据集必须有统一的 ensembl_id
sync_gene_ids(adata, gene_id_format="symbol")

# ---- 合并临床元数据（若提供）----
if CLINICAL_CSV and Path(CLINICAL_CSV).exists():
    _clinical = pd.read_csv(CLINICAL_CSV, index_col=0)
    _n_before = adata.n_obs
    adata.obs = adata.obs.join(_clinical, how="left")
    print(f"已合并临床元数据: {list(_clinical.columns)} ({_n_before} → {adata.n_obs} 细胞)")

print(f"\n加载完成: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"obs 列: {list(adata.obs.columns)}")
print(f"var 列: {list(adata.var.columns)}")


In [ ]:
# === expression_contract：建立 counts 契约 ===
# 目的：10x mtx 格式的 X 为 raw counts，建立 layers["counts"] 作为唯一权威计数层。
# layers["counts"] 是进入框架后唯一 counts 权威位置，一经建立不得被覆盖。
# SoupX 校正写入 layers["counts_soupx"]，绝不覆盖 layers["counts"]。

# 1. 建立 layers["counts"] = CSR float32，从 X 复制
adata.layers["counts"] = sp.csr_matrix(adata.X, dtype=np.float32)
print(f"layers['counts'] 已建立: shape={adata.layers['counts'].shape}, "
      f"dtype={adata.layers['counts'].dtype}, format={adata.layers['counts'].format}")

# 2. 全量验证 counts 为非负整数（raw counts 准入条件）
# 稀疏矩阵 .data 包含全部非零值，零值天然通过整数检查
_counts_data = adata.layers["counts"].data
_counts_are_nonneg = bool(np.all(_counts_data >= 0))
_counts_are_int = bool(np.all(_counts_data == np.floor(_counts_data)))
_counts_ok = bool(_counts_are_nonneg and _counts_are_int)
if not _counts_are_nonneg:
    raise ValueError(f"counts 包含负值：min={_counts_data.min():.1f}，非 raw counts")
if not _counts_are_int:
    raise ValueError(f"counts 包含非整数值，非 raw counts")
_min_count = float(_counts_data.min())
_max_count = float(_counts_data.max())

# 3. 建立 expression_contract（8 字段 schema）
adata.uns["expression_contract"] = {
    "x_scale": "raw_counts",          # X 当前尺度
    "counts_layer": "counts",         # 权威原始 counts 所在 layer
    "counts_source": "X",             # counts 直接从 X 提取（10x mtx 的 X 即 raw）
    "counts_validated": _counts_ok,   # 非负整数校验结果
    "counts_integer_check": "full",   # 全量整数校验（非 blockwise）
    "soupx_layer": None,              # SoupX layer（未运行时为 None）
    "processing_history": [
        f"01_{DATA_SOURCE_ID}: loaded from 10x mtx, X is raw counts, "
        "copied to layers['counts'] (CSR float32)"
    ],
    "stage": "01",                    # Stage 01 产物
}

# 4. 校验契约 schema（8 字段完整性 + 取值合法性）
_contract = validate_expression_contract(adata, expected_scale="raw_counts", stage="01")
print(f"\nexpression_contract 通过校验:")
print(f"  x_scale             = {_contract['x_scale']}")
print(f"  counts_layer        = {_contract['counts_layer']}")
print(f"  counts_source       = {_contract['counts_source']}")
print(f"  counts_validated    = {_contract['counts_validated']}")
print(f"  counts_integer_check= {_contract['counts_integer_check']}")
print(f"  soupx_layer         = {_contract['soupx_layer']}")
print(f"  stage               = {_contract['stage']}")
print(f"  count range: [{_min_count}, {_max_count}]")

# ---- counts checksum：SoupX 前后一致性门禁 ----
# SoupX 不覆盖 counts 的可执行门禁：保存 checksum，checkpoint cell 断言不变
_counts_checksum = float(adata.layers["counts"].sum())
_counts_checksum_nnz = int(adata.layers["counts"].nnz)
print(f"\ncounts checksum (sum={_counts_checksum:.0f}, nnz={_counts_checksum_nnz})")
del _counts_data, _counts_are_nonneg, _counts_are_int, _counts_ok, _min_count, _max_count


In [ ]:
# === 基础过滤 + QC 指标计算 ===

# ---- 细胞级基础过滤：移除基因数为 0 的细胞（空 droplet）----
# 这一步极其宽松，真正的 QC 过滤在后面 MAD-based 步骤
sc.pp.filter_cells(adata, min_genes=1)
print(f"基础细胞过滤后: {adata.n_obs:,} 细胞")

# ---- 基因级基础过滤：移除在少于 MIN_CELLS_PER_GENE 个细胞中表达的基因 ----
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
print(f"基础基因过滤后: {adata.n_vars:,} 基因 (min_cells={MIN_CELLS_PER_GENE})")

# ---- 标注线粒体基因 ----
# MT- 前缀（人类），mt- 前缀（小鼠）
adata.var["mt"] = adata.var_names.str.startswith("MT-")
print(f"线粒体基因数: {adata.var['mt'].sum()}")

# ---- QC 指标计算 ----
# 产出 obs 列：n_genes_by_counts, total_counts, pct_counts_mt
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)
print(f"QC 指标已计算: n_genes_by_counts, total_counts, pct_counts_mt")


## 样本级 QC 摘要

按 sample_id 分组统计每个样本的细胞数、基因中位数、UMI 中位数、线粒体比例中位数。

**看什么**：是否存在某个样本与其他样本差异过大（如某样本细胞数极少、或 MT% 异常偏高）。
这有助于判断是否存在批次效应、处理差异或技术故障。


In [ ]:
# === 样本级 QC 摘要表 ===
if "sample_id" in adata.obs.columns:
    sample_summary = adata.obs.groupby("sample_id").agg(
        n_cells=("n_genes_by_counts", "count"),
        median_genes=("n_genes_by_counts", "median"),
        median_umi=("total_counts", "median"),
        median_mt_pct=("pct_counts_mt", "median"),
    ).round(1)
    print(f"样本数: {len(sample_summary)}")
    display(sample_summary)
else:
    print("obs 中无 sample_id 列，跳过样本级摘要。")
    sample_summary = None


## 基线 QC 分布

绘制三个主 QC 指标的小提琴图和散点图，供在设定过滤阈值前直观判断数据质量。

**三个指标的含义**：
- `n_genes_by_counts`：每个细胞检测到的基因数。过低=空液滴或死细胞；过高=可能是双细胞
- `total_counts`：每个细胞的总 UMI 计数。与 n_genes 正相关，极低=低质量
- `pct_counts_mt`：线粒体基因占比。过高=细胞膜破损或凋亡细胞（通常 >20% 为警戒线）


In [ ]:
# === 基线 QC 小提琴图 ===
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(["n_genes_by_counts", "total_counts", "pct_counts_mt"]):
    sc.pl.violin(adata, metric, groupby="sample_id", ax=axes[i], show=False,
                rotation=45)
    axes[i].set_title(metric)
plt.tight_layout()
plt.show()

# ---- 散点图：n_genes vs MT% ----
# 用于直观判断高 MT% 细胞是否同时低基因数（典型的低质量细胞特征）
fig, ax = plt.subplots(figsize=(6, 5))
sc.pl.scatter(adata, x="n_genes_by_counts", y="pct_counts_mt", ax=ax, show=False, s=2)
ax.set_title("n_genes_by_counts vs pct_counts_mt（过滤前）")
plt.tight_layout()
plt.show()


## 自适应阈值计算（MAD-based）

MAD（median absolute deviation）是比标准差更稳健的离散度度量，对离群值不敏感。

**参数含义**：
- `N_MAD`：阈值 = 中位数 +/- N_MAD × MAD
- `PER_SAMPLE_MAD`：True=按样本独立计算阈值（推荐，各样本用自己基线）

过滤维度：
- `n_genes_by_counts`：双向过滤（过高+过低）
- `total_counts`：单向下限过滤（过低=低质量）
- `pct_counts_mt`：单项上限过滤（过高=凋亡/破损）


In [ ]:
# === 自适应阈值计算（MAD-based）===
n_before = adata.n_obs

# 过滤前统计摘要
qc_pre_stats = {
    "n_genes_by_counts": {
        "median": float(adata.obs["n_genes_by_counts"].median()),
        "mean": float(adata.obs["n_genes_by_counts"].mean()),
    },
    "total_counts": {
        "median": float(adata.obs["total_counts"].median()),
        "mean": float(adata.obs["total_counts"].mean()),
    },
    "pct_counts_mt": {
        "median": float(adata.obs["pct_counts_mt"].median()),
        "mean": float(adata.obs["pct_counts_mt"].mean()),
    },
}

if QC_STRATEGY == "adaptive":
    thresholds = {}
    for metric, direction in [
        ("n_genes_by_counts", "both"),
        ("total_counts", "lower"),
        ("pct_counts_mt", "upper"),
    ]:
        if PER_SAMPLE_MAD:
            # 按 sample_id 分组，每个样本独立计算阈值
            sample_thresholds = {}
            for sample_id in adata.obs["sample_id"].unique():
                mask = adata.obs["sample_id"] == sample_id
                vals = adata.obs.loc[mask, metric].dropna()
                if len(vals) < 10:
                    print(f"  WARNING: {sample_id} 仅有 {len(vals)} 个细胞的 {metric} 值，沿用全局阈值")
                    continue
                med = vals.median()
                mad_val = median_abs_deviation(vals, nan_policy="omit")
                lower = max(0, med - N_MAD * mad_val) if direction in ("both", "lower") else None
                upper = med + N_MAD * mad_val if direction in ("both", "upper") else None
                sample_thresholds[sample_id] = {
                    "median": round(med, 1), "mad": round(mad_val, 1),
                    "lower": round(lower, 1) if lower else None,
                    "upper": round(upper, 1) if upper else None,
                }
            thresholds[metric] = {"mode": "per_sample", "per_sample": sample_thresholds}
        else:
            # 全局 MAD
            vals = adata.obs[metric].dropna()
            med = vals.median()
            mad = median_abs_deviation(vals, nan_policy="omit")
            lower = max(0, med - N_MAD * mad) if direction in ("both", "lower") else None
            upper = med + N_MAD * mad if direction in ("both", "upper") else None
            thresholds[metric] = {
                "median": round(med, 1), "mad": round(mad, 1),
                "lower": round(lower, 1) if lower else None,
                "upper": round(upper, 1) if upper else None,
            }
elif QC_STRATEGY == "fixed":
    thresholds = {
        "n_genes_by_counts": {
            "median": None, "mad": None,
            "lower": MIN_GENES, "upper": MAX_GENES,
        },
        "total_counts": {
            "median": None, "mad": None,
            "lower": MIN_COUNTS, "upper": None,
        },
        "pct_counts_mt": {
            "median": None, "mad": None,
            "lower": None, "upper": MAX_PCT_MT,
        },
    }
    print("使用固定阈值模式")
else:
    raise ValueError(f"不支持的 QC_STRATEGY: {QC_STRATEGY}")

print(f"===== QC 阈值（{QC_STRATEGY}）=====")
print(f"  策略: {QC_STRATEGY}")
if QC_STRATEGY == "adaptive":
    print(f"  N_MAD: {N_MAD}")
    mode_str = "per-sample" if PER_SAMPLE_MAD else "global"
    print(f"  模式: {mode_str}")

if PER_SAMPLE_MAD and QC_STRATEGY == "adaptive":
    print("\n===== 各样本阈值明细 =====")
    for metric in ["n_genes_by_counts", "total_counts", "pct_counts_mt"]:
        print(f"\n--- {metric} ---")
        rows = []
        for sid, t in thresholds[metric]["per_sample"].items():
            rows.append({"sample_id": sid, **{k: v for k, v in t.items() if v is not None}})
        if rows:
            display(pd.DataFrame(rows).set_index("sample_id"))
else:
    thresh_df = pd.DataFrame({
        k: {kk: vv for kk, vv in v.items() if vv is not None}
        for k, v in thresholds.items()
    }).T
    display(thresh_df)


## N_MAD 敏感度分析

核心问题：N_MAD 太小 = 丢太多细胞（可能丢真信号）；N_MAD 太大 = 保留垃圾。
经验法则：活检组织 3-4，类器官 5-7。下面的曲线帮助你选择最佳值。

敏感度曲线与实际过滤使用同一口径（`PER_SAMPLE_MAD` 开关同时作用于曲线与过滤），
确保曲线反映的就是实际过滤行为。


In [ ]:
# === N_MAD 敏感度分析：帮助选择最优阈值 ===
# 敏感度曲线与过滤同口径——PER_SAMPLE_MAD 开关同时作用于曲线与过滤
_test_mads = sorted(set([2, 3, 4, 5, 6, 7, N_MAD, N_MAD + 1]))
sensitivity_results = []
for _nm in _test_mads:
    _keep = pd.Series(True, index=adata.obs_names)
    for metric, direction in [
        ("n_genes_by_counts", "both"),
        ("total_counts", "lower"),
        ("pct_counts_mt", "upper"),
    ]:
        if PER_SAMPLE_MAD and QC_STRATEGY == "adaptive":
            for sample_id in adata.obs["sample_id"].unique():
                smask = adata.obs["sample_id"] == sample_id
                vals = adata.obs.loc[smask, metric].dropna()
                if len(vals) < 10:
                    continue
                med = vals.median()
                mad_val = median_abs_deviation(vals, nan_policy="omit")
                if direction in ("both", "lower"):
                    _keep.loc[smask] &= adata.obs.loc[smask, metric] >= max(0, med - _nm * mad_val)
                if direction in ("both", "upper"):
                    _keep.loc[smask] &= adata.obs.loc[smask, metric] <= med + _nm * mad_val
        else:
            vals = adata.obs[metric].dropna()
            med = vals.median()
            mad_val = median_abs_deviation(vals, nan_policy="omit")
            if direction in ("both", "lower"):
                _keep &= adata.obs[metric] >= max(0, med - _nm * mad_val)
            if direction in ("both", "upper"):
                _keep &= adata.obs[metric] <= med + _nm * mad_val
    n_keep = _keep.sum()
    sensitivity_results.append({
        "N_MAD": _nm,
        "cells_kept": n_keep,
        "pct_kept": round(100 * n_keep / adata.n_obs, 1),
        "median_mt_kept": round(adata.obs.loc[_keep, "pct_counts_mt"].median(), 2),
    })

sens_df = pd.DataFrame(sensitivity_results)
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(sens_df["N_MAD"], sens_df["pct_kept"], "o-", color="steelblue", linewidth=2)
ax1.axvline(N_MAD, color="red", linestyle="--", label=f"N_MAD={N_MAD}")
ax1.set_xlabel("N_MAD")
ax1.set_ylabel("细胞保留率 (%)", color="steelblue")
ax1.set_title("N_MAD 敏感度：保留率 vs 阈值宽松度")
ax2 = ax1.twinx()
ax2.plot(sens_df["N_MAD"], sens_df["median_mt_kept"], "s--", color="orange")
ax2.set_ylabel("保留细胞的 median MT%", color="orange")
ax1.legend()
plt.tight_layout()
plt.show()
print(sens_df.to_string(index=False))
print("\n经验判据：保留 85-95% 细胞 + median MT% 不显著上升 = 合理")


In [ ]:
# === 特殊基因标记（仅标记，不移除）===

# ---- 血红蛋白基因标记 ----
# 红细胞污染/溶解的标志。仅标记不移除——消化道活检中低水平 HB 背景常见
if FLAG_HEMOGLOBIN:
    adata.var["hb"] = adata.var_names.str.upper().str.match("^HB[^P]")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["hb"], percent_top=None, log1p=False, inplace=True)
    # pct_counts_hb 已自动写入 adata.obs
    adata.obs["flag_hb"] = adata.obs["pct_counts_hb"] > 1.0
    n_hb = adata.obs["flag_hb"].sum()
    print(f"血红蛋白标记: {n_hb} 个细胞 pct_counts_hb > 1% (共 {adata.var['hb'].sum()} 个 HB 基因)")
else:
    print("血红蛋白标记已跳过 (FLAG_HEMOGLOBIN=False)")

# ---- 应激基因标记 ----
# 即时早期基因（IEGs）+ 热休克蛋白（HSPs），在组织解离过程中被机械/酶切应激诱导
# 仅标记不移除——高应激细胞可能仍包含有价值的生物学信息
if FLAG_STRESS_GENES:
    STRESS_GENES = [
        "JUN", "FOS", "JUNB", "FOSB", "ATF3", "EGR1", "ZFP36",  # IEGs
        "HSPA1A", "HSPA1B", "HSP90AA1", "HSPB1",              # HSPs
    ]
    adata.var["stress"] = adata.var_names.isin(STRESS_GENES)
    sc.pp.calculate_qc_metrics(adata, qc_vars=["stress"], percent_top=None, log1p=False, inplace=True)
    n_stress_genes = adata.var["stress"].sum()
    print(f"应激基因标记: {n_stress_genes} 个应激基因已标记")
else:
    print("应激基因标记已跳过 (FLAG_STRESS_GENES=False)")


## 环境 RNA 校正（SoupX）

环境 RNA（ambient RNA）来自裂解的细胞碎片和游离 RNA，悬浮在液滴溶液中并被随机捕获形成背景噪声。
SoupX 通过 raw droplets 矩阵（含空液滴/碎片背景）和 filtered 矩阵估计并扣除背景。

**前置条件**：需要 R 环境 + SoupX R 包 + raw_feature_bc_matrix 目录。
校正结果写入 `layers["counts_soupx"]`（CSR float32），**绝不覆盖 `layers["counts"]`**（唯一权威原始计数层）。

本 template 中 `SOUPX_ENABLED` 默认为 `False`；具备 raw droplets 数据时设为 `True`。


In [ ]:
# === 环境 RNA 校正（SoupX）—— subprocess Rscript 模式 ===
# 校正结果写入 layers["counts_soupx"]，绝不覆盖 layers["counts"]（唯一权威原始计数层）

soupx_applied = False
n_soupx_corrected = 0
soupx_needs_review = False
soupx_failed_samples = []

# 初始化 counts_soupx layer：从原始 counts 拷贝作基线
if "counts_soupx" not in adata.layers:
    adata.layers["counts_soupx"] = adata.layers["counts"].copy()
    if not sp.issparse(adata.layers["counts_soupx"]) or adata.layers["counts_soupx"].dtype != np.float32:
        adata.layers["counts_soupx"] = sp.csr_matrix(adata.layers["counts_soupx"], dtype=np.float32)
    print(f"layers['counts_soupx'] 已初始化: shape={adata.layers['counts_soupx'].shape}, "
          f"dtype={adata.layers['counts_soupx'].dtype}")

if not SOUPX_ENABLED:
    print("SoupX 已跳过: SOUPX_ENABLED=False")
    adata.uns["soupx_contract"] = {
        "enabled": False, "status": "skipped_by_user",
        "n_cells_corrected": 0, "integer_round": SOUPX_INTEGER_ROUND,
        "per_sample": {}, "failed_samples": [], "needs_review": False,
    }
elif not R_AVAILABLE:
    print("SoupX 已跳过: R 环境未就绪")
    adata.uns["soupx_contract"] = {
        "enabled": True, "status": "unavailable",
        "n_cells_corrected": 0, "integer_round": SOUPX_INTEGER_ROUND,
        "per_sample": {}, "failed_samples": [], "needs_review": False,
        "skip_reason": "R not available",
    }
else:
    # ---- R 可用且启用：执行 per-sample SoupX ----
    # 需要 manifest 提供 raw_matrix_path
    _raw_path = _manifest_data.get("input", {}).get("raw_path") if _manifest_data else None
    if not _raw_path:
        print("SoupX 已跳过: manifest 中无 raw_matrix_path")
        adata.uns["soupx_contract"] = {
            "enabled": True, "status": "skipped",
            "n_cells_corrected": 0, "integer_round": SOUPX_INTEGER_ROUND,
            "per_sample": {}, "failed_samples": [], "needs_review": False,
            "skip_reason": "no raw_matrix_path in manifest",
        }
    else:
        import subprocess, scipy.io
        soupx_script = "scripts/soupx_run.R"
        soupx_tmp = "results/_soupx_tmp"
        os.makedirs(soupx_tmp, exist_ok=True)
        raw_dir = Path(_raw_path)
        soupx_diagnostics = {}

        if "ambient_correction_applied" not in adata.obs.columns:
            adata.obs["ambient_correction_applied"] = False

        print(f"运行 SoupX per sample（raw_path={_raw_path}）...")
        for sample_id in sorted(adata.obs["sample_id"].unique()):
            sample_mask = adata.obs["sample_id"] == sample_id
            n_cells = sample_mask.sum()
            cell_ids = adata.obs_names[sample_mask]

            # 提取原始 10x barcode（移除样本前缀）
            prefix = f"{sample_id}_"
            original_barcodes = []
            for cid in cell_ids:
                if cid.startswith(prefix):
                    rest = cid[len(prefix):]
                    parts = rest.rsplit("-", 1)
                    original_barcodes.append(parts[0] if len(parts) == 2 and parts[1].isdigit() else rest)
                else:
                    original_barcodes.append(cid)

            # 定位 raw matrix 目录
            raw_sample_dir = raw_dir / sample_id / "raw_feature_bc_matrix"
            if not ((raw_sample_dir / "matrix.mtx.gz").exists() or (raw_sample_dir / "matrix.mtx").exists()):
                alt_raw = raw_dir / "raw_feature_bc_matrix"
                if (alt_raw / "matrix.mtx.gz").exists() or (alt_raw / "matrix.mtx").exists():
                    raw_sample_dir = alt_raw
                else:
                    print(f"  {sample_id}: raw_feature_bc_matrix 未找到，跳过")
                    soupx_diagnostics[sample_id] = {"status": "failed", "reason": "raw matrix not found"}
                    soupx_failed_samples.append(sample_id)
                    soupx_needs_review = True
                    continue

            try:
                # 导出 filtered matrix 子集
                _filtered_mtx = soupx_tmp + f"/{sample_id}_filtered"
                os.makedirs(_filtered_mtx, exist_ok=True)
                _sub = adata[sample_mask].copy()
                _counts_sub = _sub.layers["counts"]
                scipy.io.mmwrite(f"{_filtered_mtx}/matrix.mtx", _counts_sub)
                pd.Series(original_barcodes).to_csv(f"{_filtered_mtx}/barcodes.tsv", index=False, header=False)
                pd.Series(_sub.var_names).to_csv(f"{_filtered_mtx}/genes.tsv", index=False, header=False)

                # 调用 Rscript
                _result = subprocess.run(
                    [RSCRIPT_BIN, "--vanilla", soupx_script,
                     str(raw_sample_dir), _filtered_mtx, soupx_tmp + f"/{sample_id}"],
                    capture_output=True, text=True, timeout=120,
                )
                if _result.returncode == 0:
                    # 读回校正后矩阵并写回 counts_soupx
                    _corrected = scipy.io.mmread(f"{soupx_tmp}/{sample_id}_corrected.mtx")
                    if _corrected.shape == _counts_sub.shape:
                        _corrected_csr = sp.csr_matrix(_corrected, dtype=np.float32)
                        if SOUPX_INTEGER_ROUND:
                            _corrected_csr.data = np.round(_corrected_csr.data)
                        adata.layers["counts_soupx"][sample_mask.to_numpy()] = _corrected_csr
                        adata.obs.loc[sample_mask, "ambient_correction_applied"] = True
                        n_soupx_corrected += n_cells
                        soupx_applied = True
                        print(f"  {sample_id}: SoupX 完成 ({n_cells} 细胞)")
                    else:
                        raise ValueError(f"校正矩阵 shape 不匹配: {_corrected.shape} vs {_counts_sub.shape}")
                else:
                    raise RuntimeError(_result.stderr.strip() or f"Rscript exit code {_result.returncode}")
            except Exception as e:
                print(f"  {sample_id}: SoupX 失败 - {e}")
                soupx_diagnostics[sample_id] = {"status": "failed", "reason": str(e)[:200]}
                soupx_failed_samples.append(sample_id)
                soupx_needs_review = True

        # 写入 soupx_contract
        adata.uns["soupx_contract"] = {
            "enabled": True,
            "status": "partial" if soupx_failed_samples else "success",
            "n_cells_corrected": n_soupx_corrected,
            "integer_round": SOUPX_INTEGER_ROUND,
            "per_sample": soupx_diagnostics,
            "failed_samples": soupx_failed_samples,
            "needs_review": soupx_needs_review,
        }
        print(f"\nSoupX 完成: {n_soupx_corrected} 细胞已校正"
              + (f", {len(soupx_failed_samples)} 样本失败" if soupx_failed_samples else ""))
        # 更新 expression_contract 中的 soupx_layer
        if soupx_applied:
            adata.uns["expression_contract"]["soupx_layer"] = "counts_soupx"
            adata.uns["expression_contract"]["processing_history"].append(
                f"01_{DATA_SOURCE_ID}: SoupX applied, wrote to layers['counts_soupx']"
            )


## 双细胞鉴定（Scrublet，三态定级）

双细胞（doublet）是两个细胞被误包在同一个液滴中测序。其基因表达是两种细胞类型的混合，
会干扰细胞类型注释和差异表达分析。Scrublet 通过模拟双细胞来预测每个液滴是否为 doublet。

**三态定级策略**（singlet / uncertain / doublet）：
- `doublet_score > HIGH` → **doublet**（高置信，默认排除）
- `LOW <= doublet_score <= HIGH` → **uncertain**（边界，默认保留）
- `doublet_score < LOW` → **singlet**（高置信单细胞，保留）

只标记不物理删除细胞。01 保存完整对象，02 按 `doublet_include` 构建整合对象。

如果 manifest 声明原作者已做 doublet removal（`preprocessing_done` 含 `doublet_removal`），
则自动跳过。


In [ ]:
# === 双细胞鉴定：per-sample scrublet（三态定级）===
# 只标记不物理删除细胞。高置信 doublet 默认排除（doublet_include=False）

import scrublet as scr
try:
    _scrublet_version = scr.__version__
except AttributeError:
    try:
        from importlib.metadata import version
        _scrublet_version = version("scrublet")
    except Exception:
        _scrublet_version = "unknown"

# ---- obs 列初始化（fresh-kernel 安全）----
adata.obs["doublet_score"] = np.nan
adata.obs["doublet_class"] = "singlet"
adata.obs["predicted_doublet"] = False
adata.obs["doublet_include"] = True

# ---- manifest-driven skip check ----
_pp_done = _manifest_data.get("preprocessing_done", []) if _manifest_data else []
_qc_override = _manifest_data.get("qc_overrides", {}).get("doublet_removal", {}) if _manifest_data else {}
skip_doublet = False
skip_reason = None
if "doublet_removal" in _pp_done:
    skip_doublet = True
    skip_reason = "原作者已去除双细胞（preprocessing_done 含 doublet_removal）"
elif _qc_override.get("skip"):
    skip_doublet = True
    skip_reason = _qc_override.get("reason", "qc_overrides 声明跳过")

# ---- per-sample 诊断容器 ----
doublet_diagnostics = {}
per_sample_thresholds = {}
doublet_needs_review = False
needs_review_reasons = []

# ---- doublet 输入层选择（SoupX 校正后优先）----
_dbl_source_layer = adata.uns["expression_contract"].get("soupx_layer") or "counts"
print(f"doublet 检测输入层: layers['{_dbl_source_layer}']")

if skip_doublet:
    print(f"双细胞鉴定已跳过: {skip_reason}")
    doublet_method = "skipped"
elif not DOUBLET_DETECTION:
    print("DOUBLET_DETECTION=False，所有细胞标记 singlet")
    doublet_method = "not_run"
elif EXPECTED_DOUBLET_RATE is None:
    print("EXPECTED_DOUBLET_RATE=None，未运行 scrublet")
    doublet_method = "not_run"
else:
    doublet_method = "scrublet"
    print(f"运行 Scrublet per sample（expected_doublet_rate={EXPECTED_DOUBLET_RATE}，输入层={_dbl_source_layer}）...")

    for sample_id in sorted(adata.obs["sample_id"].unique()):
        sample_mask = adata.obs["sample_id"] == sample_id
        n_sample = sample_mask.sum()

        # 细胞数不足 → needs_review
        if n_sample < DOUBLET_MIN_CELLS:
            reason = f"too_few_cells: {sample_id} ({n_sample} < {DOUBLET_MIN_CELLS})"
            needs_review_reasons.append(reason)
            doublet_needs_review = True
            print(f"  {sample_id}: {n_sample} 细胞（< {DOUBLET_MIN_CELLS}），全部标记 singlet")
            doublet_diagnostics[sample_id] = {
                "n_cells": n_sample, "n_singlet": n_sample,
                "n_uncertain": 0, "n_doublet": 0,
                "needs_review": True, "reason": reason,
            }
            per_sample_thresholds[sample_id] = {"low": None, "high": None}
            continue

        # 子集 copy + scrublet 检测
        sub = adata[sample_mask].copy()
        scrub_input = sub.layers[_dbl_source_layer]
        scrub = scr.Scrublet(
            scrub_input, expected_doublet_rate=EXPECTED_DOUBLET_RATE,
            random_state=RANDOM_SEED,
        )
        doublet_scores, _ = scrub.scrub_doublets()
        auto_threshold = scrub.threshold_

        # 定阈值：优先级 DOUBLET_SCORE_THRESHOLD > DOUBLET_SCORE_HIGH > auto
        if DOUBLET_SCORE_THRESHOLD is not None:
            threshold_high = DOUBLET_SCORE_THRESHOLD
        elif DOUBLET_SCORE_HIGH is not None:
            threshold_high = DOUBLET_SCORE_HIGH
        else:
            threshold_high = auto_threshold if auto_threshold is not None else 0.25
            if auto_threshold is None:
                reason = f"scrublet_threshold_none: {sample_id} (auto_threshold=None, fallback 0.25)"
                needs_review_reasons.append(reason)
                doublet_needs_review = True

        threshold_low = DOUBLET_SCORE_LOW if DOUBLET_SCORE_LOW is not None else threshold_high * 0.5

        # 三态定级
        adata.obs.loc[sample_mask, "doublet_score"] = doublet_scores
        adata.obs.loc[sample_mask, "doublet_class"] = "uncertain"
        adata.obs.loc[(sample_mask) & (doublet_scores < threshold_low), "doublet_class"] = "singlet"
        adata.obs.loc[(sample_mask) & (doublet_scores > threshold_high), "doublet_class"] = "doublet"

        # 转为 Categorical
        adata.obs["doublet_class"] = pd.Categorical(
            adata.obs["doublet_class"], categories=["singlet", "uncertain", "doublet"]
        )

        # 诊断
        n_dbl = (doublet_scores > threshold_high).sum()
        n_unc = ((doublet_scores >= threshold_low) & (doublet_scores <= threshold_high)).sum()
        n_sgl = (doublet_scores < threshold_low).sum()
        dbl_rate = n_dbl / n_sample

        per_sample_thresholds[sample_id] = {
            "low": round(threshold_low, 4), "high": round(threshold_high, 4),
            "auto": round(auto_threshold, 4) if auto_threshold is not None else None,
        }
        doublet_diagnostics[sample_id] = {
            "n_cells": n_sample, "n_singlet": n_sgl,
            "n_uncertain": n_unc, "n_doublet": n_dbl,
            "doublet_rate": round(dbl_rate, 4),
            "threshold_low": round(threshold_low, 4),
            "threshold_high": round(threshold_high, 4),
            "needs_review": False,
        }

        # 异常高比例告警
        if DOUBLET_RATE_ALERT_HIGH is not None and dbl_rate > DOUBLET_RATE_ALERT_HIGH:
            reason = f"high_doublet_rate: {sample_id} ({dbl_rate:.1%} > {DOUBLET_RATE_ALERT_HIGH:.1%})"
            needs_review_reasons.append(reason)
            doublet_needs_review = True
            doublet_diagnostics[sample_id]["needs_review"] = True

        print(f"  {sample_id}: singlet={n_sgl} uncertain={n_unc} doublet={n_dbl} "
              f"(rate={dbl_rate:.1%}, high={threshold_high:.4f})")


In [ ]:
# === 双细胞三态收尾：include 派生 + doublet_contract 持久化 ===
# 只标记不物理删除细胞（01 保存完整对象；02 按 doublet_include 构建整合对象）

# ---- include 列派生 ----
adata.obs["doublet_include"] = True
adata.obs.loc[adata.obs["doublet_class"] == "doublet", "doublet_include"] = False
if not DOUBLET_UNCERTAIN_INCLUDE:
    adata.obs.loc[adata.obs["doublet_class"] == "uncertain", "doublet_include"] = False

# 向后兼容：predicted_doublet = 仅高置信 doublet
adata.obs["predicted_doublet"] = adata.obs["doublet_class"] == "doublet"

# ---- 全局汇总 ----
n_dbl = int((adata.obs["doublet_class"] == "doublet").sum())
n_unc = int((adata.obs["doublet_class"] == "uncertain").sum())
n_sgl = int((adata.obs["doublet_class"] == "singlet").sum())
n_excluded = int((~adata.obs["doublet_include"]).sum())
n_doublets_total = n_dbl  # 供 QC 报告使用

print(f"双细胞三态汇总: singlet={n_sgl}  uncertain={n_unc}  doublet={n_dbl}")
print(f"排除细胞数: {n_excluded} (高置信 doublet)")
if n_unc > 0:
    print(f"uncertain 保留: {n_unc} ({'纳入' if DOUBLET_UNCERTAIN_INCLUDE else '排除'})")

# ---- doublet_contract 持久化到 uns ----
adata.uns["doublet_contract"] = {
    "method": doublet_method if "doublet_method" in dir() else "not_run",
    "method_version": _scrublet_version if "_scrublet_version" in dir() else "unknown",
    "expected_doublet_rate": EXPECTED_DOUBLET_RATE,
    "per_sample_thresholds": per_sample_thresholds,
    "per_sample_diagnostics": doublet_diagnostics,
    "n_singlet": n_sgl, "n_uncertain": n_unc, "n_doublet": n_dbl,
    "n_excluded": n_excluded,
    "uncertain_include": DOUBLET_UNCERTAIN_INCLUDE,
    "needs_review": doublet_needs_review,
    "needs_review_reasons": needs_review_reasons,
    "skip_reason": skip_reason if ("skip_doublet" in dir() and skip_doublet) else None,
}


In [ ]:
# === 细胞周期评分（Tirosh 2015 marker genes）===
# 细胞周期阶段（G1/S/G2M）是重要的技术协变量，下游可据此回归
# 基因表直接写在 cell 内，不进 src——分析逻辑的可见性优先于代码复用

if not CELL_CYCLE_SCORING:
    print("细胞周期评分已跳过 (CELL_CYCLE_SCORING=False)")
else:
    print("===== 细胞周期评分 =====")

    s_genes = [
        "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1",
        "UNG", "GINS2", "MCM6", "CDCA7", "DTL", "PRIM1", "UHRF1",
        "MLF1IP", "HELLS", "RFC2", "RPA2", "NASP", "RAD51AP1",
        "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7", "POLD3", "MSH2",
        "ATAD2", "RAD51", "RRM2", "CDC45", "CDC6", "EXO1", "TIPIN",
        "DSCC1", "BLM", "CASP8AP2", "USP1", "CLSPN", "POLA1",
        "CHAF1B", "BRIP1", "E2F8",
    ]
    g2m_genes = [
        "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A",
        "NDC80", "CKS2", "NUF2", "CKS1B", "MKI67", "TMPO", "CENPF",
        "TACC3", "FAM64A", "SMC4", "CCNB2", "CKAP2L", "CKAP2",
        "AURKB", "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1",
        "KIF20B", "HJURP", "CDCA3", "HN1", "CDC20", "TTK", "CDC25C",
        "KIF2C", "RANGAP1", "NCAPD2", "DLGAP5", "CDCA2", "CDCA8",
        "ECT2", "KIF23", "HMMR", "AURKA", "PSRC1", "ANLN", "LBR",
        "CKAP5", "CENPE", "CTCF", "NEK2", "G2E3", "GAS2L3", "CBX5",
        "CENPA",
    ]

    # 只保留数据中实际存在的基因
    s_genes_found = [g for g in s_genes if g in adata.var_names]
    g2m_genes_found = [g for g in g2m_genes if g in adata.var_names]
    print(f"S 期基因: {len(s_genes_found)}/{len(s_genes)} 命中")
    print(f"G2M 期基因: {len(g2m_genes_found)}/{len(g2m_genes)} 命中")

    sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes_found, g2m_genes=g2m_genes_found)
    print(f"细胞周期分布:")
    print(adata.obs["phase"].value_counts())


## QC 过滤

根据上一步得出的阈值过滤低质量细胞。

**不在此步移除的**：
- 双细胞（`predicted_doublet`）：仅标记，供下游聚类后决定
- 血红蛋白高表达细胞：仅标记（`flag_hb`）
- 应激高表达细胞：仅标记（`pct_counts_stress`）


In [ ]:
# === QC 过滤：应用阈值 ===
print("===== QC 过滤 =====")
cells_before = adata.n_obs
print(f"过滤前细胞数: {cells_before:,}")

# 保存过滤前状态供交叉诊断使用
if "flag_hb" in adata.obs.columns or "pct_counts_stress" in adata.obs.columns:
    obs_pre_filter = adata.obs.copy()

keep = pd.Series(True, index=adata.obs_names)

if QC_STRATEGY == "adaptive":
    for metric, direction in [
        ("n_genes_by_counts", "both"),
        ("total_counts", "lower"),
        ("pct_counts_mt", "upper"),
    ]:
        if PER_SAMPLE_MAD:
            for sample_id in adata.obs["sample_id"].unique():
                smask = adata.obs["sample_id"] == sample_id
                t = thresholds[metric].get("per_sample", {}).get(sample_id, {})
                if not t:
                    continue
                if t.get("lower") is not None and direction in ("both", "lower"):
                    keep.loc[smask] &= adata.obs.loc[smask, metric] >= t["lower"]
                if t.get("upper") is not None and direction in ("both", "upper"):
                    keep.loc[smask] &= adata.obs.loc[smask, metric] <= t["upper"]
        else:
            t = thresholds[metric]
            if t.get("lower") is not None and direction in ("both", "lower"):
                keep &= adata.obs[metric] >= t["lower"]
            if t.get("upper") is not None and direction in ("both", "upper"):
                keep &= adata.obs[metric] <= t["upper"]
elif QC_STRATEGY == "fixed":
    for metric, t in thresholds.items():
        if t.get("lower") is not None:
            keep &= adata.obs[metric] >= t["lower"]
        if t.get("upper") is not None:
            keep &= adata.obs[metric] <= t["upper"]

n_filtered = (~keep).sum()
print(f"过滤: {n_filtered} 个细胞 ({100 * n_filtered / cells_before:.1f}%)")
for metric in ["n_genes_by_counts", "total_counts", "pct_counts_mt"]:
    if QC_STRATEGY == "adaptive" and PER_SAMPLE_MAD:
        # per-sample 模式下按样本统计
        for sid in adata.obs["sample_id"].unique():
            smask = adata.obs["sample_id"] == sid
            n_pass = (keep & smask).sum()
            n_total = smask.sum()
            if n_total - n_pass > 0:
                print(f"  {sid}.{metric}: {n_pass}/{n_total} 通过")
    else:
        n_pass = keep.sum()
        break
    if QC_STRATEGY == "adaptive" and not PER_SAMPLE_MAD:
        break

adata = adata[keep].copy()
cells_after = adata.n_obs
print(f"过滤后细胞数: {cells_after:,} (移除 {cells_before - cells_after:,}, "
      f"保留 {100 * cells_after / cells_before:.1f}%)")


In [ ]:
# === 过滤前后 QC 对比 ===
# 复刻过滤前的 QC 图，供直观判断过滤效果
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, metric in enumerate(["n_genes_by_counts", "total_counts", "pct_counts_mt"]):
    # 过滤前（从保存的 pre-filter obs 中取）
    if "obs_pre_filter" in dir():
        _pre = obs_pre_filter[[metric]].copy()
        _pre["filter"] = "过滤前"
        _post = adata.obs[[metric]].copy()
        _post["filter"] = "过滤后"
        _combined = pd.concat([_pre, _post])
        for j, filt in enumerate(["过滤前", "过滤后"]):
            _vals = _combined.loc[_combined["filter"] == filt, metric].dropna()
            axes[j, i].hist(_vals, bins=50, color="steelblue" if j == 0 else "darkorange", alpha=0.7)
            axes[j, i].set_title(f"{metric} ({filt})")
            if j == 0:
                axes[j, i].axvline(
                    thresholds[metric].get("lower") if QC_STRATEGY != "adaptive" or not PER_SAMPLE_MAD else None,
                    color="red", linestyle="--", alpha=0.5)
                axes[j, i].axvline(
                    thresholds[metric].get("upper") if QC_STRATEGY != "adaptive" or not PER_SAMPLE_MAD else None,
                    color="red", linestyle="--", alpha=0.5)
    else:
        axes[0, i].hist(adata.obs[metric], bins=50, color="steelblue", alpha=0.7)
        axes[0, i].set_title(f"{metric} (过滤后)")
        axes[1, i].axis("off")
plt.tight_layout()
plt.show()

# 过滤后 violin
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(["n_genes_by_counts", "total_counts", "pct_counts_mt"]):
    sc.pl.violin(adata, metric, groupby="sample_id", ax=axes[i], show=False, rotation=45)
    axes[i].set_title(f"{metric} (过滤后)")
plt.tight_layout()
plt.show()


In [ ]:
# === QC 报告摘要 ===
# 汇总本次 QC 的全部参数与结果，写入 adata.uns["qc_report"]
n_removed = cells_before - cells_after

qc_report = {
    "strategy": QC_STRATEGY,
    "n_mad": N_MAD if QC_STRATEGY == "adaptive" else None,
    "per_sample_mad": PER_SAMPLE_MAD if QC_STRATEGY == "adaptive" else False,
    "n_before": int(cells_before),
    "n_after": int(cells_after),
    "n_removed": int(n_removed),
    "pct_removed": round(100 * n_removed / cells_before, 1),
    "thresholds": {
        k: {
            "lower": v.get("lower"), "upper": v.get("upper"),
        } for k, v in thresholds.items()
    },
    "soupx": {"enabled": SOUPX_ENABLED, "applied": soupx_applied,
               "n_corrected": n_soupx_corrected},
    "doublet": {
        "method": doublet_method if "doublet_method" in dir() else "not_run",
        "n_singlet": n_sgl, "n_uncertain": n_unc, "n_doublet": n_dbl,
        "n_excluded": n_excluded,
    },
    "cell_cycle": CELL_CYCLE_SCORING,
}
adata.uns["qc_report"] = qc_report

print("===== QC 报告摘要 =====")
print(f"策略: {QC_STRATEGY}")
if QC_STRATEGY == "adaptive":
    print(f"N_MAD: {N_MAD} (per_sample={PER_SAMPLE_MAD})")
print(f"细胞: {cells_before:,} → {cells_after:,} (移除 {n_removed:,}, {100*n_removed/cells_before:.1f}%)")
print(f"SoupX: 启用={SOUPX_ENABLED}, 校正={n_soupx_corrected} 细胞")
print(f"双细胞: singlet={n_sgl} uncertain={n_unc} doublet={n_dbl} 排除={n_excluded}")
print(f"细胞周期评分: {CELL_CYCLE_SCORING}")


In [ ]:
# === Checkpoint：run_contract + 写入 h5ad + schema 校验 ===
# 先计算所有可见门禁；只有到保存阶段才占用 RUN_ID

# ---- 基因 ID 轴一致性断言 ----
# 大小写混用在 merge 时会导致 inner join 基因交集意外坍塌
_gene_names = list(adata.var_names)
_upper_count = sum(1 for g in _gene_names if g[0].isupper()) if _gene_names else 0
_lower_count = sum(1 for g in _gene_names if g[0].islower()) if _gene_names else 0
_total = len(_gene_names)
if _upper_count > 0 and _lower_count > 0:
    raise ValueError(
        f"基因 ID 轴不一致：{_upper_count} 个大写首字母基因 + {_lower_count} 个小写首字母基因。"
        f"请统一基因名大小写后再进入 merge。"
    )
print(f"基因 ID 轴一致: {_total} 个基因，统一{'大写' if _upper_count > 0 else '小写'}首字母")

# ---- expression_contract 硬门禁 ----
_contract_ok = True
_contract_error = None
try:
    _contract = validate_expression_contract(adata, expected_scale="raw_counts", stage="01")
except (KeyError, ValueError) as _e:
    _contract_ok = False
    _contract_error = str(_e)

_layers_counts_ok = (
    "counts" in adata.layers
    and sp.issparse(adata.layers["counts"])
    and adata.layers["counts"].dtype == np.float32
)

# ---- counts_layer_unchanged 门禁 ----
_counts_layer_unchanged = True
if globals().get("_counts_checksum") is not None and "counts" in adata.layers:
    _curr_sum = float(adata.layers["counts"].sum())
    _curr_nnz = int(adata.layers["counts"].nnz)
    _counts_layer_unchanged = (
        abs(_curr_sum - globals().get("_counts_checksum", _curr_sum)) < 1
        and _curr_nnz == globals().get("_counts_checksum_nnz", _curr_nnz)
    )
    if not _counts_layer_unchanged:
        print(f"CRITICAL layers['counts'] 已被修改! "
              f"sum {globals().get('_counts_checksum')} → {_curr_sum}, "
              f"nnz {globals().get('_counts_checksum_nnz')} → {_curr_nnz}")

# ---- soupx_layer_consistent 门禁 ----
_soupx_layer_consistent = True
_soupx_layer = adata.uns["expression_contract"].get("soupx_layer")
if _soupx_layer == "counts_soupx":
    _soupx_layer_consistent = (
        "counts_soupx" in adata.layers
        and adata.layers["counts_soupx"].dtype == np.float32
    )

# ---- hard postconditions ----
_source_values = sorted(map(str, adata.obs["source_dataset"].dropna().unique())) if "source_dataset" in adata.obs.columns else []
hard_postconditions = {
    "non_empty": adata.n_obs > 0 and adata.n_vars > 0,
    "x_sparse_float32": sp.issparse(adata.X) and adata.X.dtype == np.float32,
    "expression_contract": _contract_ok,
    "layers_counts_csr_f32": _layers_counts_ok,
    "counts_layer_unchanged": _counts_layer_unchanged,
    "soupx_layer_consistent": _soupx_layer_consistent,
    "source_dataset_unique": len(_source_values) == 1 and not adata.obs["source_dataset"].isna().any(),
    "source_matches_manifest": _source_values == [str(source_dataset)],
}

# ---- doublet postconditions（仅在 doublet 已运行时验证）----
_hd = "doublet_class" in adata.obs.columns
hard_postconditions["doublet_columns_present"] = all(
    c in adata.obs for c in ["doublet_score", "doublet_class", "doublet_include", "predicted_doublet"]
) if _hd else True
hard_postconditions["doublet_contract_present"] = "doublet_contract" in adata.uns if _hd else True
hard_postconditions["doublet_class_valid"] = (
    set(adata.obs["doublet_class"].dropna().unique()) <= {"singlet", "uncertain", "doublet"}
) if _hd else True
hard_postconditions["doublet_include_consistent"] = (
    not adata.obs.loc[adata.obs["doublet_class"] == "doublet", "doublet_include"].any()
) if _hd else True

if not _contract_ok:
    print(f"WARNING expression_contract 校验失败: {_contract_error}")

# ---- needs_review 综合判定 ----
_doublet_needs_review = bool(globals().get("doublet_needs_review", False))
_soupx_needs_review = bool(globals().get("soupx_needs_review", False))
_needs_review = _doublet_needs_review or _soupx_needs_review

stage_status = determine_stage_status(
    {}, hard_postconditions, needs_review=_needs_review, allow_no_required_methods=True
)

# ---- run_contract：manifest + checkpoint ----
effective_parameters = snapshot_effective_parameters(
    globals(), exclude=("RSCRIPT_BIN", "R_AVAILABLE"), path_root=Path(_root)
)
runtime_provenance = collect_runtime_provenance(
    _root, ("anndata", "scanpy", "numpy", "pandas", "scipy", "scrublet")
)
manifest_sha256 = sha256_file(MANIFEST_YAML) if _manifest_exists else "no_manifest"
run_paths = prepare_run(RUN_ROOT, RUN_ID)
manifest_payload = {
    "run_id": RUN_ID, "stage": "01_qcd", "stage_status": stage_status.value,
    "source_dataset": str(source_dataset),
    "inputs": [{"path": MANIFEST_YAML, "sha256": manifest_sha256}] if _manifest_exists else [],
    "effective_parameters": effective_parameters, "runtime_provenance": runtime_provenance,
    "hard_postconditions": hard_postconditions,
}

if stage_status.value == "FAILED":
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise RuntimeError(f"Stage 01 FAILED: {hard_postconditions}")

adata.uns["stage"] = "01_qcd"
adata.uns["status"] = stage_status.value
adata.uns["upstream"] = [MANIFEST_YAML] if _manifest_exists else [RAW_MATRIX_DIR]
adata.uns["version"] = OUTPUT_VERSION
adata.uns["run_id"] = RUN_ID

draft_checkpoint = run_paths.draft_dir / OUTPUT_FILENAME
try:
    adata.write_h5ad(draft_checkpoint, compression="lzf")
    checkpoint_sha256 = sha256_file(draft_checkpoint)
except Exception as error:
    draft_checkpoint.unlink(missing_ok=True)
    manifest_payload["stage_status"] = "FAILED"
    manifest_payload["failure"] = {"type": type(error).__name__, "message": str(error)}
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise

manifest_payload["checkpoint"] = {"path": OUTPUT_FILENAME, "sha256": checkpoint_sha256}
atomic_write_json(run_paths.manifest_path, manifest_payload)

# ---- 分支处理 ----
if stage_status.value == "NEEDS_REVIEW":
    if _hd:
        manifest_payload["doublet_summary"] = {
            "needs_review": _doublet_needs_review,
            "n_excluded": int((~adata.obs["doublet_include"]).sum()),
            "n_doublet": int((adata.obs["doublet_class"] == "doublet").sum()),
            "n_uncertain": int((adata.obs["doublet_class"] == "uncertain").sum()),
            "n_singlet": int((adata.obs["doublet_class"] == "singlet").sum()),
            "reasons": globals().get("needs_review_reasons", []),
        }
    _sc = adata.uns.get("soupx_contract", {})
    if _soupx_needs_review or _sc.get("needs_review"):
        manifest_payload["soupx_summary"] = {
            "status": _sc.get("status"),
            "n_cells_corrected": _sc.get("n_cells_corrected", 0),
            "failed_samples": _sc.get("failed_samples", []),
            "needs_review": bool(_sc.get("needs_review", False)),
        }
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    OUTPUT_PATH = str(run_paths.draft_dir / OUTPUT_FILENAME)
    print(f"[NEEDS_REVIEW] draft checkpoint 已写入 {OUTPUT_PATH}")
    reasons = []
    if _soupx_needs_review:
        reasons.append(f"SoupX 部分/全部样本失败: {_sc.get('failed_samples', [])}")
    if _doublet_needs_review:
        reasons.append(f"doublet 检测异常: {globals().get('needs_review_reasons', [])}")
    for r in reasons:
        print(f"  - {r}")
    print(f"  请 PI 审阅诊断表后决定是否手动 promote 或调整参数重跑。")
elif stage_status.value == "FAILED":
    raise RuntimeError(f"Stage 01 FAILED: {hard_postconditions}")
else:
    OUTPUT_PATH = str(promote_run(run_paths))
    print(f"OK 提升 {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes)")

# ---- per_dataset schema 校验 ----
_schema_result = validate_per_dataset_output(adata)
if not _schema_result["passed"]:
    print("per_dataset schema 校验 FAILED:")
    for e in _schema_result["errors"]:
        print(f"  [ERROR] {e}")
else:
    print("per_dataset schema 校验 PASSED")
if _schema_result["warnings"]:
    for w in _schema_result["warnings"]:
        print(f"  [WARN] {w}")

# ---- 释放内存 ----
gc.collect()
del adata; gc.collect()
print("内存已释放。")


## 回跑与新版本

### 调参回跑

1. bump `OUTPUT_VERSION`（v1 → v2）→ `RUN_ID` 自动更新
2. 修改需调整的 QC 参数或方法开关
3. 从头运行全部 cell
4. 新的 `RUN_ID` 会创建独立目录，不覆盖旧 run

### RUN_ID 与目录结构

每个 RUN_ID 在 `results/runs/` 下创建独立目录：

```
results/runs/
  01-dataset_id-v1/
    draft/manifest.json    # 执行事实验证记录
    draft/01_dataset_id_v1.h5ad
    promoted/              # promote 后出现
```

### 上下游关系

- **上游**：cellranger 输出的 `filtered_feature_bc_matrix` 目录
- **下游**：`02_merged.ipynb` 读取所有 per-dataset h5ad 进行整合
- **契约**：产出的 h5ad 必须通过 `validate_per_dataset_output()` 校验才能进入 merge

### 常见问题

- **RUN_ID 已存在**：`prepare_run()` 会 raise，确认要覆盖则先手动删除旧目录
- **NEEDS_REVIEW**：检查 doublet 诊断表或 SoupX 失败日志，PI 判断后可手动 bump 版本
- **schema 校验失败**：检查 `doublet_class` dtype 是否为 Categorical、`layers["counts"]` 是否已建立
